In [63]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq 

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

DATA_FILE_PATH = os.getenv("DATA_FILE_PATH", "data/hr_policy.txt")

### DATA INGESTATION


In [64]:
loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents = loader.load()

print("=== Loaded Documents ===")
print(documents)

=== Loaded Documents ===
[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of 

## LANGCHAIN DOCUMENT
Langchain processess everything in form of documents

# DOCUMENTS

PAGE CONTENT - The Actual data
METADATA - Extra information about the data



In [65]:
len(documents)

1

In [66]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [67]:
print(documents[0].metadata)

{'source': 'data/hr_policy.txt'}


In [68]:
print("=== Document words length ===")
print(len(documents[0].page_content))

=== Document words length ===
2598


SPLITTING OUR DATA


In [69]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print("=== Splitted Documents ===")
print(chunks)
len(chunks)

=== Splitted Documents ===
[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable 

9

NOW EACH SPLITED CHUNK IS A DOCUMENT - Page content and metadata

In [70]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data/hr_policy.txt'}


In [71]:
print(chunks[2])

page_content='2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.' metadata={'source': 'data/hr_policy.txt'}


In [72]:
print(chunks[8])

page_content='8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.' metadata={'source': 'data/hr_policy.txt'}


EMDEBDD OUR DATA

In [73]:
from langchain_community.document_loaders import TextLoader
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en", api_key=jina_api_key)

print("=== Embeddings Model ===")
print(embeddings_model)



=== Embeddings Model ===
session=<requests.sessions.Session object at 0x00000223D858EF90> model_name='jina-embeddings-v2-base-en' jina_api_key=None


In [74]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings_model)

print("=== Vectorstore ===")
print(vectorstore.index.ntotal)

=== Vectorstore ===
9


In [75]:
test_query = "What is the company's policy on remote work?"
docs = vectorstore.similarity_search(test_query, k=3)

print("=== Similarity Search query ===",test_query)

for i,match in enumerate(docs):
    print(f"=== Match {i+1} ===")
    print(match.page_content)

=== Similarity Search query === What is the company's policy on remote work?
=== Match 1 ===
2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.
=== Match 2 ===
5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.
=== Match 3 ===
8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed wit

DATA RETRIVAL


In [76]:
from langchain_groq import ChatGroq 

llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.9, api_key=groq_api_key)


print("=== LLM ===", llm.model_name)


=== LLM === openai/gpt-oss-120b


In [77]:
test_response = llm.invoke("What is the company's policy on remote work?")
test_response.content

'I’m happy to help, but I need a little more information first. Which company’s remote‑work policy are you interested in learning about? If you let me know the name (or the industry, if you’re looking for a typical example), I can provide the relevant details or point you to where you can find the official policy.'

AI AGENT




3 . 

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - No Memory


In [ ]:
from langchain.agents import create_agent

retrieve_tool = vectorstore.as_retriever(search_kwargs={"k": 3})

def search_hr_pilocy(question:str)->str:
    """Search the HR policy documents for the answer to the question."""
    matching_chuncks = retrieve_tool.invoke(question)
    return "\n\n".join([chunk.page_content for chunk in matching_chuncks])

hr_assistance = create_agent(
    model = llm, 
    tools = [search_hr_pilocy], 
    system_prompt="You are an HR assistant. You will answer questions about the company's policies based on the provided documents. If the answer is not in the documents, respond with 'I don't know.'")

print("=== HR ASSISTANCE IS READY ===")
response = hr_assistance.invoke({
    "message":[
        {"role": "user", "content": "What is the company's policy on remote work?"}
    ]
})

answer = response['messages'][-1].content
print("=== HR ASSISTANCE RESPONSE ===")
print(answer)
